In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    'dim_client': RAW_DIR / 'dim_client.xlsx',
    'dim_product': RAW_DIR / 'dim_product.xlsx',
    'fact_sales': RAW_DIR / 'fact_sales.xlsx',
}

RAW_FILES

{'dim_client': WindowsPath('C:/Users/bochr/OneDrive/Documents/projet Medis/data/raw/dim_client.xlsx'),
 'dim_product': WindowsPath('C:/Users/bochr/OneDrive/Documents/projet Medis/data/raw/dim_product.xlsx'),
 'fact_sales': WindowsPath('C:/Users/bochr/OneDrive/Documents/projet Medis/data/raw/fact_sales.xlsx')}

In [2]:
import subprocess
import unicodedata


def clean_column_name(col):
    """Convert raw column names to lowercase_with_underscores."""
    col = str(col).strip().lower()
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('ascii')
    col = re.sub(r'[^0-9a-zA-Z]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return col


def make_unique_columns(columns):
    """Avoid duplicate column names after cleaning names."""
    seen = {}
    unique = []
    for col in columns:
        if col not in seen:
            seen[col] = 0
            unique.append(col)
        else:
            seen[col] += 1
            unique.append(f'{col}_{seen[col]}')
    return unique


def load_excel(path):
    """Load the first sheet of an Excel file.

    If Excel or OneDrive locks the original file, this tries to read a temporary
    copy. If that also fails, close the workbook and run the notebook again.
    """
    try:
        excel_file = pd.ExcelFile(path)
        print(f'{path.name}: using sheet {excel_file.sheet_names[0]}')
        return pd.read_excel(path, sheet_name=excel_file.sheet_names[0])
    except PermissionError:
        temp_path = PROCESSED_DIR / f'_temporary_copy_{path.name}'
        try:
            command = f"Copy-Item -LiteralPath '{path}' -Destination '{temp_path}' -Force"
            subprocess.run(['powershell', '-NoProfile', '-Command', command], check=True)
            excel_file = pd.ExcelFile(temp_path)
            print(f'{path.name}: original file is locked, using a temporary copy')
            return pd.read_excel(temp_path, sheet_name=excel_file.sheet_names[0])
        except Exception as error:
            raise PermissionError(f'Close {path.name} in Excel/OneDrive and run again.') from error
        finally:
            if temp_path.exists():
                temp_path.unlink()


def standardize_columns(df):
    """Clean column names before renaming them to English."""
    df = df.copy()
    df.columns = make_unique_columns([clean_column_name(c) for c in df.columns])
    return df


def clean_text_columns(df):
    """Strip spaces in text columns and turn blank strings into missing values."""
    df = df.copy()
    text_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in text_cols:
        df[col] = df[col].astype('string').str.strip()
        df[col] = df[col].replace({'': np.nan, 'nan': np.nan, 'None': np.nan, '<NA>': np.nan})
    return df


def print_change(step, before, after):
    """Print how many rows changed after a cleaning step."""
    print(f'{step}: removed {before - after:,} rows | {before:,} -> {after:,}')


def require_columns(df, required_columns, table_name):
    """Stop early if an expected column is missing."""
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f'{table_name} is missing these columns: {missing}')

In [3]:
# Load the three raw tables.
raw_tables = {name: load_excel(path) for name, path in RAW_FILES.items()}

for name, df in raw_tables.items():
    print(f'{name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns')

dim_client.xlsx: using sheet Feuil1
dim_product.xlsx: using sheet Feuil1
fact_sales.xlsx: using sheet Feuil1
dim_client: 1,173 rows, 15 columns
dim_product: 2,882 rows, 17 columns
fact_sales: 295,338 rows, 8 columns


In [4]:
def clean_fact_sales(df):
    df = standardize_columns(df)

    fact_rename = {
        'prix_vente': 'unit_price',
        'numero_facture': 'invoice_id',
        'reference_produit': 'product_id',
        'code_client': 'client_id',
        'date_facture': 'invoice_date',
        'montant_total_vente': 'revenue',
        'quantite': 'quantity',
        'societe': 'company',
    }
    df = df.rename(columns=fact_rename)
    require_columns(df, fact_rename.values(), 'fact_sales')
    df = clean_text_columns(df)

    print('FACT_SALES')
    print(f'Starting shape: {df.shape}')

    before = len(df)
    df = df.drop_duplicates().copy()
    print_change('Exact duplicate rows', before, len(df))

    df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')

    # Revenue: fill nulls with 0 so zero-price rows stay explicit before we drop them.
    revenue_nulls = df['revenue'].isna().sum()
    print(f'Revenue nulls before fill: {revenue_nulls:,}')
    df['revenue'] = df['revenue'].fillna(0)

    # Remove returns / credit notes for gross-sales forecasting.
    before = len(df)
    return_mask = (df['quantity'] < 0) | (df['revenue'] < 0)
    df = df[~return_mask].copy()
    print_change('Return / credit-note rows', before, len(df))

    # Zero unit_price means a non-sale transaction, so drop it.
    zero_price_rows = (df['unit_price'] == 0).sum()
    print(f'Zero-price rows before filtering: {zero_price_rows:,}')
    before = len(df)
    df = df[df['unit_price'] > 0].copy()
    print_change('Zero-price rows', before, len(df))

    # Quantity: drop nulls and zeros only.
    before = len(df)
    df = df[df['quantity'].notna() & (df['quantity'] != 0)].copy()
    print_change('Null or zero quantity rows (data quality)', before, len(df))

    # Zero revenue: ambiguous (not a sale, not a clear return) ? drop.
    before = len(df)
    df = df[df['revenue'] != 0].copy()
    print_change('Zero revenue rows (ambiguous ? neither sale nor return)', before, len(df))

    print(f'\nFinal shape: {df.shape}')
    print('Missing values after cleaning:')
    display(df.isna().sum().to_frame('missing_values'))
    display(df.head())
    return df


fact_sales = clean_fact_sales(raw_tables['fact_sales'])


FACT_SALES
Starting shape: (295338, 8)
Exact duplicate rows: removed 2,518 rows | 295,338 -> 292,820
Revenue nulls before fill: 0
Return / credit-note rows: removed 1,525 rows | 292,820 -> 291,295
Zero-price rows before filtering: 753
Zero-price rows: removed 753 rows | 291,295 -> 290,542
Null or zero quantity rows (data quality): removed 0 rows | 290,542 -> 290,542
Zero revenue rows (ambiguous ? neither sale nor return): removed 0 rows | 290,542 -> 290,542

Final shape: (290542, 8)
Missing values after cleaning:


,missing_values
unit_price,0
invoice_id,0
product_id,0
client_id,0
invoice_date,0
revenue,0
quantity,0
company,0


,unit_price,invoice_id,product_id,client_id,invoice_date,revenue,quantity,company
0,14.373,N2500075,ARA016T030,TUN00094,2025-02-06,140.8554,10,NEAPOLIS
1,8.044,M2500453,PF005900006,TUN00125,2025-02-27,472.9872,60,MEDIS
2,8.310,M2502537,PF005300001,TUN00100,2025-12-04,97.7256,12,MEDIS
4,18.488,M2600817,PF013500010,TUN00125,2026-04-27,362.3648,20,MEDIS
5,7.850,M2302535,PF002700010,TUN00060,2023-12-15,184.6320,24,MEDIS


## 2. Clean dim_product

We keep the useful product columns for forecasting and remove columns that are
mostly descriptors or not needed for this project.

**Latest changes (applied here):**
- Dropped `sales_model` — it was correlated with a column in another table, so keeping it would cause redundancy.
- Filled every empty cell in `pack_coefficient` with `1` — an empty pack coefficient means we treat each unit as a pack of 1.


In [5]:
def clean_dim_product(df):
    df = standardize_columns(df)

    product_rename = {
        'reference_produit': 'product_id',
        'desgination_produit': 'product_name',
        'famille': 'product_family',
        'dosage': 'dosage',
        'modele': 'sales_model',        # we rename it first, then drop it below
        'forme': 'pack_size',
        'activite': 'therapeutic_area',
        'nature': 'product_category',
        'dci': 'inn',
        'societe': 'company',
        'mc_indus': 'manufacturing_category',
        'forme_galenique': 'drug_form',
        'incoterm': 'incoterm',
        'forme_pharmaceutique': 'pharmaceutical_form',
        'activite_eng': 'therapeutic_area_eng',
        'ancien_code': 'old_product_id',
        'coef': 'pack_coefficient',
    }
    df = df.rename(columns=product_rename)
    require_columns(df, ['product_id', 'product_name', 'product_family', 'therapeutic_area', 'pack_coefficient'], 'dim_product')
    df = clean_text_columns(df)

    print('DIM_PRODUCT')
    print(f'Starting shape: {df.shape}')
    print(f'Exact duplicate rows: {df.duplicated().sum():,}')

    # All the columns we don't need for forecasting.
    # sales_model (originally "Modele") is also dropped here — it was found to
    # be correlated with a column in another table, making it redundant.
    columns_to_drop = [
        'dosage',
        'pack_size',
        'product_category',
        'inn',
        'manufacturing_category',
        'drug_form',
        'incoterm',
        'pharmaceutical_form',
        'therapeutic_area_eng',
        'old_product_id',
        'sales_model',   # ← dropped: correlated with a column in another table
        'company',       # ← dropped: same as the company column in another table (originally "Societe")
    ]
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
    print(f'After dropping unused columns: {df.shape}')

    df['product_family'] = df['product_family'].str.upper()
    df['therapeutic_area'] = df['therapeutic_area'].str.upper()

    # ── Therapeutic area: map all 31 raw spellings to 7 clean English categories ──
    area_map = {
        'ONCO, IMMUNO': 'Oncology & Immunology',
        'IMMUNO & ONCOLOGY': 'Oncology & Immunology',
        'CARCINOLOGIE': 'Oncology & Immunology',
        'CARCINOLOGIE / CAPECITABINE': 'Oncology & Immunology',

        'CARDIO, METAB. & RESPIRATOIRE': 'Cardio, Metabolic & Respiratory',
        'CARDIO, METAB & RESPIRATORY': 'Cardio, Metabolic & Respiratory',
        'CARDIOLOGIE': 'Cardio, Metabolic & Respiratory',
        'CARDIOLOGIE & ANGIOLOGIE': 'Cardio, Metabolic & Respiratory',

        'GASTRO, TRAUMATO & INFECTIOLOGIE': 'Gastro, Trauma & Infectiology',
        'GASTRO., TRAUMATO. & INFECTIOLOGY': 'Gastro, Trauma & Infectiology',
        'GASTROENTEROLOGIE': 'Gastro, Trauma & Infectiology',
        'INFECTIOLOGIE': 'Gastro, Trauma & Infectiology',
        'INFECTIOLOGIE / CIPROFLOXACINE': 'Gastro, Trauma & Infectiology',
        'INFECTIOLOGIE/CLAR': 'Gastro, Trauma & Infectiology',

        'REA, RADIO & DENTAIRE': 'Rea, Radio & Dental',
        'FORMES STERILES': 'Rea, Radio & Dental',
        'ANESTHESIOLOGIE': 'Rea, Radio & Dental',

        'SNC & SEXOLOGIE': 'CNS & Sexology',
        'NEUROLOGIE': 'CNS & Sexology',
        'PSYCHIATRIE': 'CNS & Sexology',

        'COMPL. ALIMENTAIRES': 'Food Supplements',
        'CHOLÉCALCIFÉROL': 'Food Supplements',

        'ANTI-INFLAMMATOIRES': 'Others',
        'HEMATHOLOGIE': 'Others',
        'METABOLISME ET NUTRITION': 'Others',
        'ALLERGOLOGIE': 'Others',
        'ALLERGOLOGIE / DESLORATADINE': 'Others',
        'OPHTALMOLOGIE': 'Others',
        'IMMUNOSUPPRESSION': 'Others',
        'UROLOGIE-NÉPHROLOGIE': 'Others',
        'AUTRES': 'Others',
    }
    df['therapeutic_area_clean'] = df['therapeutic_area'].map(area_map).fillna('Others')

    # ── Product family: fix known typos only, leave everything else as-is ─────────
    family_map = {
        'ANASTRAZOLE': 'ANASTROZOLE',
        'ANASTRAZOL': 'ANASTROZOLE',
        'SUGAMEDEX': 'SUGAMMADEX',
        'PERMETRE XED': 'PEMETREXED',
        'DECETAXEL': 'DOCETAXEL',
        'ATRACRURIUM': 'ATRACURIUM',
    }
    df['product_family_clean'] = df['product_family'].replace(family_map)

    # Drop the old columns now that we have clean standardized versions
    df = df.drop(columns=['therapeutic_area', 'product_family'])
    print('Dropped old therapeutic_area and product_family columns (keeping clean versions only)')

    df['pack_coefficient'] = pd.to_numeric(df['pack_coefficient'], errors='coerce')

    # Fill missing pack_coefficient with 1 — a missing value means one unit per pack.
    before_fill = df['pack_coefficient'].isna().sum()
    df['pack_coefficient'] = df['pack_coefficient'].fillna(1)
    print(f'Pack coefficient: filled {before_fill:,} missing values with 1')
    display(df[['product_family_clean']].value_counts(dropna=False).head(15).to_frame('rows'))
    display(df[['therapeutic_area_clean']].value_counts(dropna=False).head(20).to_frame('rows'))
    display(df.isna().sum().to_frame('missing_values'))
    display(df.head())
    return df


dim_product = clean_dim_product(raw_tables['dim_product'])

DIM_PRODUCT
Starting shape: (2882, 17)
Exact duplicate rows: 0
After dropping unused columns: (2882, 5)
Dropped old therapeutic_area and product_family columns (keeping clean versions only)
Pack coefficient: filled 698 missing values with 1


,rows
product_family_clean,
AUTRES,698
ENOXA,120
IPPROTON,103
LOSAR,84
IRYS,66
FLUKAS,64
STOPALGIC,59
ARALEAD,57
MEDICAINE,55


,rows
therapeutic_area_clean,
Others,849
"Gastro, Trauma & Infectiology",754
"Cardio, Metabolic & Respiratory",672
"Rea, Radio & Dental",220
Oncology & Immunology,148
CNS & Sexology,134
Food Supplements,105


,missing_values
product_id,0
product_name,0
pack_coefficient,0
therapeutic_area_clean,0
product_family_clean,0


,product_id,product_name,pack_coefficient,therapeutic_area_clean,product_family_clean
0,ABI250T120,ABIRATERONE NEAPOLIS 250 MG,120.0,Oncology & Immunology,ABIRATERONE
1,ABI250T120AP,ABIRATERONE 250 MG ACCES PROG,120.0,Oncology & Immunology,ABIRATERONE
2,ABI250T120LBN,ABIRATERONE NEAPOLIS 250MG LBN,120.0,Oncology & Immunology,ABIRATERONE
3,ACA050T090D01,ACARBOSE50MG B90 DEV,1.0,Others,AUTRES
4,ADE015T030,"ADEX LP 1,5 MG CP B30",30.0,"Cardio, Metabolic & Respiratory",ADEX


## 3. Clean dim_client

We keep the client attributes needed for segmentation and forecasting, then standardize the small spelling/capitalization issues.

In [6]:
def clean_dim_client(df):
    df = standardize_columns(df)

    client_rename = {
        'codeclient': 'client_id',
        'client': 'client_name',
        'type': 'client_type',
        'marche': 'market_type',
        'pays': 'country_fr',
        'region': 'region',
        'continent': 'continent',
        'domaine': 'client_channel',
        'critere': 'market_segment',
        'section': 'section_code',
        'txcommsion': 'commission_rate',
        'ico': 'ico',
        'country': 'country_en',
        'condition_paiement': 'payment_terms',
        'mode_facturation': 'invoicing_mode',
    }
    df = df.rename(columns=client_rename)
    require_columns(df, ['client_id', 'client_name', 'client_type', 'market_type', 'region', 'client_channel', 'commission_rate'], 'dim_client')
    df = clean_text_columns(df)

    print('DIM_CLIENT')
    print(f'Starting shape: {df.shape}')
    print(f'Exact duplicate rows: {df.duplicated().sum():,}')

    columns_to_drop = [
        'country_fr',
        'continent',
        'market_segment',
        'section_code',
        'ico',
        'country_en',
        'payment_terms',
        'invoicing_mode',
    ]
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
    print(f'After dropping unused columns: {df.shape}')

    df['client_type'] = df['client_type'].str.upper()
    df['client_type'] = df['client_type'].replace({'CLIENTS GROUPE': 'CLIENT GROUPE'})

    df['market_type'] = df['market_type'].str.upper()
    df['region'] = df['region'].str.upper()
    df['region'] = df['region'].replace({'REGION': 'AUTRE', 'AUTRES': 'AUTRE'})
    df['client_channel'] = df['client_channel'].str.upper()

    df['commission_rate'] = df['commission_rate'].astype('string').str.replace(',', '.', regex=False)
    df['commission_rate'] = pd.to_numeric(df['commission_rate'], errors='coerce')
    print(f'Non-zero commission rates: {(df["commission_rate"].fillna(0) != 0).sum():,}')

    print(f'Duplicate client_id values: {df.duplicated(subset=["client_id"]).sum():,}')
    display(df[['client_type']].value_counts(dropna=False).to_frame('rows'))
    display(df[['market_type']].value_counts(dropna=False).to_frame('rows'))
    display(df[['region']].value_counts(dropna=False).to_frame('rows'))
    display(df[['client_channel']].value_counts(dropna=False).to_frame('rows'))
    display(df.isna().sum().to_frame('missing_values'))
    display(df.head())
    return df


dim_client = clean_dim_client(raw_tables['dim_client'])

DIM_CLIENT
Starting shape: (1173, 15)
Exact duplicate rows: 0
After dropping unused columns: (1173, 7)
Non-zero commission rates: 42
Duplicate client_id values: 0


,rows
client_type,
CLIENT GROUPE,1056
PHARMACIE CENTRALE,27
AUTRE,26
CLINIQUE,26
SOUS-TRAITANTS,17
CHU,16
MARCHÉ TURQUIE OFF,3
ALL PHARMA DISTRI,1
SALMA PARAPHARM,1


,rows
market_type,
EXPORT,625
LOCAL,505
AUTRE,26
SOUS-TRAITANTS,17


,rows
region,
MAGHREB,746
AFRIQUE SUBSAHARIENNE,238
MOYEN ORIENT,104
AUTRE,39
EUROPE DE L'EST,14
EUROPE CENTRALE,11
EUROPE,9
AMERIQUE LATINE,8
EUROPE DE SUD,4


,rows
client_channel,
OFFICINE,997
HOPITAUX,150
AUTRE,26


,missing_values
client_id,0
client_name,1
client_type,0
market_type,0
region,0
client_channel,0
commission_rate,0


,client_id,client_name,client_type,market_type,region,client_channel,commission_rate
0,AFNSS000,AFRIQUE NOIRE SUBSAHARIENNE,AUTRE,AUTRE,AUTRE,AUTRE,0.0
1,AL-SHEFA,AL-SHEFA INTERNATIONAL DRUG STORE,CLIENT GROUPE,EXPORT,MOYEN ORIENT,OFFICINE,0.0
2,ALFAWASSER,ALFA WASSERMAN,CLIENT GROUPE,LOCAL,MAGHREB,OFFICINE,0.0
3,ALG000,MARCHE ALGERIE,CLIENT GROUPE,EXPORT,MAGHREB,OFFICINE,0.0
4,ALG000HO,MARCHE HOSPITALIER - ALGERIE,CLIENT GROUPE,EXPORT,MAGHREB,HOPITAUX,0.0


In [7]:
# Save each cleaned table alone in data/processed.
clean_tables = {
    'fact_sales': fact_sales,
    'dim_product': dim_product,
    'dim_client': dim_client,
}

for name, df in clean_tables.items():
    output_path = PROCESSED_DIR / f'clean_{name}.csv'
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'Saved {name}: {output_path} | shape={df.shape}')

Saved fact_sales: C:\Users\bochr\OneDrive\Documents\projet Medis\data\processed\clean_fact_sales.csv | shape=(290542, 8)
Saved dim_product: C:\Users\bochr\OneDrive\Documents\projet Medis\data\processed\clean_dim_product.csv | shape=(2882, 5)
Saved dim_client: C:\Users\bochr\OneDrive\Documents\projet Medis\data\processed\clean_dim_client.csv | shape=(1173, 7)


## 4. Merge after cleaning

The fact table is the main table, so we use left joins. Product IDs are not perfectly unique in the product dimension, so we keep the first row per product ID before merging to avoid multiplying sales rows.

In [8]:
client_key = 'client_id'
product_key = 'product_id'

print('Dimension key checks before merge')
print(f'dim_client duplicate client_id: {dim_client.duplicated(subset=[client_key]).sum():,}')
print(f'dim_product duplicate product_id: {dim_product.duplicated(subset=[product_key]).sum():,}')

# Keep one row per key to prevent the merge from creating extra fact rows.
dim_client_unique = dim_client.drop_duplicates(subset=[client_key], keep='first').copy()
dim_product_unique = dim_product.drop_duplicates(subset=[product_key], keep='first').copy()

print(f'dim_client rows before/after key deduplication: {len(dim_client):,} -> {len(dim_client_unique):,}')
print(f'dim_product rows before/after key deduplication: {len(dim_product):,} -> {len(dim_product_unique):,}')

merged = fact_sales.merge(
    dim_client_unique,
    on=client_key,
    how='left',
    suffixes=('', '_client')
)

merged = merged.merge(
    dim_product_unique,
    on=product_key,
    how='left',
    suffixes=('', '_product')
)

# Fill missing values in categorical columns
# Note: therapeutic_area_clean uses 'Others' (not 'AUTRES') to be consistent with the mapping
for col in ['therapeutic_area_clean', 'product_family_clean']:  # sales_model was dropped from dim_product
    if col in merged.columns:
        if col == 'therapeutic_area_clean':
            merged[col] = merged[col].fillna('Others')
        else:
            merged[col] = merged[col].fillna('AUTRES')

Dimension key checks before merge
dim_client duplicate client_id: 0
dim_product duplicate product_id: 101
dim_client rows before/after key deduplication: 1,173 -> 1,173
dim_product rows before/after key deduplication: 2,882 -> 2,781


In [9]:
# Validate the merge.
if len(merged) == len(fact_sales):
    print('OK: merged row count stayed the same.')
else:
    print('WARNING: merged row count changed. Investigate duplicate keys before modeling.')

print(f'Rows without a matching client: {merged["client_name"].isna().sum():,}')
print(f'Rows without a matching product: {merged["product_name"].isna().sum():,}')

missing_after_merge = (merged.isna().mean() * 100).sort_values(ascending=False).head(30)
display(missing_after_merge.to_frame('missing_percent'))

OK: merged row count stayed the same.
Rows without a matching client: 0
Rows without a matching product: 215


,missing_percent
pack_coefficient,0.074
product_name,0.074
invoice_id,0.000
unit_price,0.000
product_id,0.000
client_id,0.000
quantity,0.000
company,0.000
invoice_date,0.000
revenue,0.000


In [10]:
# Save the final merged table for EDA and modeling.
merged_output = PROCESSED_DIR / 'merged_sales.csv'
merged.to_csv(merged_output, index=False, encoding='utf-8-sig')
print(f'Saved merged table: {merged_output} | shape={merged.shape}')

pickle_output = PROCESSED_DIR / 'merged_sales.pkl'
merged.to_pickle(pickle_output)
print(f'Saved pickle version: {pickle_output}')

Saved merged table: C:\Users\bochr\OneDrive\Documents\projet Medis\data\processed\merged_sales.csv | shape=(290542, 18)
Saved pickle version: C:\Users\bochr\OneDrive\Documents\projet Medis\data\processed\merged_sales.pkl


## What to tell your supervisor

- I kept the raw Excel files unchanged.
- I cleaned `fact_sales`, `dim_product`, and `dim_client` separately before saving them.
- I renamed the important columns to clear English names from the project schema.
- **Returns and credit notes are now removed** from `fact_sales` instead of being retained. Rows with negative quantity or negative revenue are filtered out so the fact table matches a gross-sales forecasting setup.
- I still drop: exact duplicate rows, zero-price rows, rows with null or zero quantity, and rows with zero revenue (ambiguous -- not a clear sale or return).
- Discount features (`computed_rev`, `discount_gap`, `discount_pct`) can now be computed on the remaining sale rows without special handling for returns.
- I standardized product/client category text values so the same category is not split into several spellings.
- **`dim_product` updates:**
  - Created `therapeutic_area_clean`: standardized 31 raw spellings into 7 clean English categories.
  - Created `product_family_clean`: fixed known typos in product family names (e.g., ANASTRAZOLE → ANASTROZOLE).
  - Dropped the old `therapeutic_area` and `product_family` columns to keep only the standardized clean versions.
  - Filled all empty `pack_coefficient` cells with `1` (a missing value means one unit per pack).
  - Dropped `sales_model` (correlated with a column in another table).
- I saved the three cleaned tables in `data/processed`.
- The final merged dataset includes only the standardized clean columns for better analytics and forecasting.
